In [13]:
from openai import OpenAI
import os
import re
from dotenv import load_dotenv

load_dotenv('../.env')

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ OpenAI client ready")

✅ OpenAI client ready


In [14]:
def generate_loan_email(applicant_data: dict, prediction: int) -> str:
    decision_text = "APPROVED" if prediction == 1 else "DENIED"

    prompt = f"""
You are a professional loan officer at a financial institution.
Write a formal, empathetic email to a loan applicant informing them 
of their loan decision.

APPLICANT DETAILS:
- Loan Amount Requested: ${applicant_data.get('loan_amount', 'N/A')} thousand
- Loan Term: {applicant_data.get('loan_term', 'N/A')} months
- Employment: {'Self-Employed' if applicant_data.get('self_employed') == 1 else 'Salaried'}
- Education: {'Graduate' if applicant_data.get('education') == 1 else 'Not Graduate'}
- Credit History: {'Good' if applicant_data.get('credit_history') == 1.0 else 'Poor'}
- Property Area: {applicant_data.get('property_area', 'N/A')}

DECISION: {decision_text}

STRICT RULES YOU MUST FOLLOW:
1. Do NOT invent or mention any specific interest rates
2. Do NOT invent any repayment schedules or monthly payment amounts
3. Do NOT make any promises beyond the decision stated above
4. Do NOT mention any fees, charges, or penalties not provided
5. Keep the email under 200 words
6. Use a professional, empathetic tone
7. End with next steps appropriate to the decision
8. Sign off as "The LoanWise AI Team"

Write only the email — no subject line, no extra commentary.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",       # cheap, fast, more than good enough
        max_tokens=500,
        messages=[
            {
                "role": "system",
                "content": "You are a professional loan officer writing formal customer emails."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.choices[0].message.content

In [15]:
def validate_email(email_text: str, applicant_data: dict) -> dict:
    """
    Checks the generated email against guardrail rules.
    
    Returns:
        dict with 'passed' (bool) and 'violations' (list of strings)
    """
    violations = []
    
    # Guardrail 1: Check for invented interest rates
    interest_patterns = [
        r'\d+\.?\d*\s*%\s*(interest|rate|APR|per annum)',
        r'(interest|rate)\s+of\s+\d+\.?\d*\s*%',
        r'APR\s+of\s+\d+'
    ]
    for pattern in interest_patterns:
        if re.search(pattern, email_text, re.IGNORECASE):
            violations.append("GUARDRAIL FAILED: Email contains invented interest rate")
            break
    
    # Guardrail 2: Check for invented monthly payment amounts
    payment_patterns = [
        r'monthly\s+payment\s+of\s+\$[\d,]+',
        r'pay\s+\$[\d,]+\s+per\s+month',
        r'installment\s+of\s+\$[\d,]+'
    ]
    for pattern in payment_patterns:
        if re.search(pattern, email_text, re.IGNORECASE):
            violations.append("GUARDRAIL FAILED: Email contains invented payment amount")
            break
    
    # Guardrail 3: Check email is not too long (hallucination risk)
    word_count = len(email_text.split())
    if word_count > 300:
        violations.append(f"GUARDRAIL FAILED: Email too long ({word_count} words, max 300)")
    
    # Guardrail 4: Check correct decision is mentioned
    decision_word = "approved" if applicant_data.get('prediction') == 1 else "denied"
    opposite_word = "denied" if applicant_data.get('prediction') == 1 else "approved"
    
    if opposite_word.lower() in email_text.lower() and decision_word.lower() not in email_text.lower():
        violations.append("GUARDRAIL FAILED: Email mentions wrong decision")
    
    # Guardrail 5: Must be signed by LoanWise AI Team
    if "loanwise" not in email_text.lower():
        violations.append("GUARDRAIL FAILED: Missing required signature")
    
    return {
        "passed": len(violations) == 0,
        "violations": violations,
        "word_count": word_count
    }

In [16]:
def generate_validated_email(applicant_data: dict, prediction: int, max_retries: int = 3) -> dict:
    """
    Full pipeline: generate email → validate → retry if needed.
    
    Returns:
        dict with email text, validation result, and attempt count
    """
    applicant_data['prediction'] = prediction
    
    for attempt in range(1, max_retries + 1):
        print(f"Attempt {attempt}/{max_retries}: Generating email...")
        
        email_text = generate_loan_email(applicant_data, prediction)
        validation = validate_email(email_text, applicant_data)
        
        if validation['passed']:
            print(f"✅ Email passed guardrails on attempt {attempt}")
            return {
                "status": "SENT",
                "email": email_text,
                "attempts": attempt,
                "validation": validation
            }
        else:
            print(f"❌ Attempt {attempt} failed guardrails:")
            for v in validation['violations']:
                print(f"   - {v}")
    
    # All attempts failed — flag for human review
    print("🚨 All attempts failed — flagging for human review")
    return {
        "status": "HUMAN_REVIEW",
        "email": None,
        "attempts": max_retries,
        "validation": validation
    }

In [17]:
# Sample approved applicant
sample_applicant = {
    "loan_amount": 128,
    "loan_term": 360,
    "self_employed": 0,
    "education": 1,
    "credit_history": 1.0,
    "property_area": "Urban"
}

result = generate_validated_email(sample_applicant, prediction=1)

print("\n" + "=" * 50)
print(f"STATUS: {result['status']}")
print(f"ATTEMPTS: {result['attempts']}")
print("=" * 50)
if result['email']:
    print("\nGENERATED EMAIL:")
    print("-" * 50)
    print(result['email'])

Attempt 1/3: Generating email...
✅ Email passed guardrails on attempt 1

STATUS: SENT
ATTEMPTS: 1

GENERATED EMAIL:
--------------------------------------------------
Dear [Applicant's Name],

Thank you for your recent application for the loan of $128,000 with a term of 360 months. We appreciate the time and effort you invested in the application process.

After careful review of your financial profile, we are pleased to inform you that your loan application has been approved. Your solid credit history and stable employment have contributed positively to this decision.

We understand that securing a loan is an important step toward your financial goals, and we are here to support you throughout the process. 

The next steps involve gathering any necessary documentation to finalize your loan agreement. A representative from our team will reach out to you shortly to discuss the details and guide you through the finalization process.

Once again, thank you for choosing us for your financi

In [18]:
# Sample denied applicant
denied_applicant = {
    "loan_amount": 200,
    "loan_term": 360,
    "self_employed": 1,
    "education": 0,
    "credit_history": 0.0,
    "property_area": "Rural"
}

result_denied = generate_validated_email(denied_applicant, prediction=0)

print("\n" + "=" * 50)
print(f"STATUS: {result_denied['status']}")
print(f"ATTEMPTS: {result_denied['attempts']}")
print("=" * 50)
if result_denied['email']:
    print("\nGENERATED EMAIL:")
    print("-" * 50)
    print(result_denied['email'])

Attempt 1/3: Generating email...
✅ Email passed guardrails on attempt 1

STATUS: SENT
ATTEMPTS: 1

GENERATED EMAIL:
--------------------------------------------------
Dear [Applicant's Name],

Thank you for your recent application for a loan of $200,000 with a term of 360 months. We appreciate the time and effort you put into your request and understand that this is an important step for you.

After careful consideration, we regret to inform you that your loan application has been denied. This decision was primarily influenced by the current evaluation of your credit history and employment status. We recognize that this may be disappointing news, and we empathize with the effort you have made in seeking financial support.

If you would like to discuss alternative options or ways to improve your eligibility in the future, we encourage you to reach out. Our team is here to assist you in any way we can.

Thank you again for considering us for your lending needs. We wish you the best in yo